# TASK5 股票涨跌分类模型

本 Notebook 使用 `model_data_stock.csv` 文件已有因子，严格按 `Date` 时间顺序划分训练集和测试集，训练逻辑回归、决策树、随机森林三个分类模型，并查看 AUC、ROC、混淆矩阵等结果。

## Step 1：导入依赖与参数配置

逻辑回归默认使用 `penalty='elasticnet'`、`solver='saga'`；随机森林默认 `max_depth=10`、`n_estimators=100`。

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, HTML, IFrame
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'run_task5_analysis.py').exists():
    NOTEBOOK_DIR = Path('/Users/wuhao/Desktop/MyData/MyWorld/MyWork/量化/BA工作坊课程/BA工作坊每周作业/task5')
sys.path.insert(0, str(NOTEBOOK_DIR))
import run_task5_analysis as task5

task5.TRAIN_RATIO = 0.70
task5.RANDOM_STATE = 42
task5.LR_PENALTY = 'elasticnet'
task5.LR_SOLVER = 'saga'
task5.LR_L1_RATIO = 0.5
task5.LR_C = 1.0
task5.LR_MAX_ITER = 5000
task5.DT_MAX_DEPTH = 10
task5.RF_MAX_DEPTH = 10
task5.RF_N_ESTIMATORS = 100

## Step 2：加载数据

读取 CSV，解析日期，并按 `Date`、`Code` 排序。

In [ ]:
df = task5.load_data()
display(df.head())
print('数据规模:', df.shape)
print('日期范围:', df['Date'].min().date(), '至', df['Date'].max().date())
print('股票数量:', df['Code'].nunique())

## Step 3：数据质量检查

检查字段类型、缺失值、重复记录、标签分布和无穷大值。

In [ ]:
quality = task5.build_quality_tables(df)
display(quality['overview'])
display(quality['label_dist'])
display(quality['missing'].head(20))
display(quality['infinite_counts'].head(20))

## Step 4：特征工程

完全使用文件已有因子：排除 `Date`、`Code`、`Y` 后，其余可转换为数值型的列进入模型。

In [ ]:
X, y, feature_names, feature_summary = task5.prepare_features(df)
print('入模因子数量:', len(feature_names))
display(feature_summary)
display(X.head())

## Step 5：严格按时间顺序划分训练集/测试集

前 70% 日期作为训练集，后 30% 日期作为测试集。同一个日期的全部股票样本放在同一侧，避免未来数据泄漏。

In [ ]:
X_train, X_test, y_train, y_test, meta_train, meta_test, split_summary = task5.split_by_time(df, X, y, task5.TRAIN_RATIO)
display(split_summary)
display(pd.DataFrame({'训练集标签分布': y_train.value_counts(normalize=True), '测试集标签分布': y_test.value_counts(normalize=True)}))

## Step 6：训练三个模型

训练步骤：缺失值只用训练集中位数填充；逻辑回归额外做标准化；决策树和随机森林不做标准化。

In [ ]:
metrics, roc_data, confusion_data, feature_importance, models = task5.train_and_evaluate(X_train, X_test, y_train, y_test, feature_names)
parameters = task5.model_parameters()
display(parameters)
display(metrics)

## Step 7：查看模型评价

根据 AUC、F1-score、Precision、Recall 和混淆矩阵，对三个模型分别评价。

In [ ]:
comments = task5.evaluate_models_text(metrics)
for name, comment in comments.items():
    display(HTML(f'<h3>{name}</h3><p>{comment}</p>'))

## Step 8：生成 ECharts HTML 看板

看板包含可交互 ROC 曲线、指标对比、混淆矩阵、特征重要性和模型评价。

In [ ]:
analysis = {
    'df': df,
    'quality': quality,
    'feature_names': feature_names,
    'feature_summary': feature_summary,
    'split_summary': split_summary,
    'metrics': metrics,
    'roc_data': roc_data,
    'confusion_data': confusion_data,
    'feature_importance': feature_importance,
    'parameters': parameters,
    'comments': comments,
}
task5.write_outputs(analysis)
task5.generate_dashboard(analysis)
print('HTML 看板已生成:', task5.DASHBOARD_PATH)
IFrame(src=str(task5.DASHBOARD_PATH), width='100%', height=820)